In [7]:
import torch
import torch.nn as nn
import random
from matplotlib import pyplot as plt

# dataset
def synthetic_data(w, b, num_examples):
    x = torch.normal(0,1,(num_examples, len(w)))
    y = torch.matmul(x,w) + b
    y += torch.normal(0,0.01,y.shape)
    
    return x, y.reshape((-1,1))

true_w = torch.tensor([2,-3.4])
true_b = 4.5

features, labels = synthetic_data(true_w, true_b, 1000)

def data_iter(batch_size, features, labels):
    num_examples = len(features)
    indices = list(range(num_examples))
    random.shuffle(indices)
    for i in range(0, num_examples, batch_size):
        batch_indices = indices[i:min(i+batch_size,num_examples)]
        yield features[batch_indices], labels[batch_indices]


# 定义网络结构和参数初始化
class LinearNet(nn.Module):
    def __init__(self):
        super(LinearNet, self).__init__()
        # self.w = nn.Parameter(torch.normal(0,1,size=(2,1),requires_grad=True))   # 需要进行梯度更新的参数需要额外使用nn.Parameter实例化
        # self.b = nn.Parameter(torch.zeros(1,requires_grad=True))
        
        self.w = nn.Parameter(torch.zeros(size=(2,1),requires_grad=True))   # 需要进行梯度更新的参数需要额外使用nn.Parameter实例化
        self.b = nn.Parameter(torch.zeros(1,requires_grad=True))
    
    def forward(self, x):
        y = torch.matmul(x,self.w) + self.b
        return y

model = LinearNet()

# 定义损失函数
Loss = nn.MSELoss()

# 定义优化函数
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)

# 定义循环结构
epochs = 3
lr = 0.04
batch_size = 8

for epoch in range(epochs):
    for batch_data, batch_label in data_iter(batch_size,features,labels):
        loss = Loss(model(batch_data),batch_label)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    with torch.no_grad():
        train_l = Loss(model(features),labels)
        print(f'epoch {epoch + 1}, loss {float(train_l.mean()):f}')



epoch 1, loss 0.263672
epoch 2, loss 0.002157
epoch 3, loss 0.000115


In [3]:
# 方法一： 保存模型结构和参数权重,
torch.save(model, "model_best.pth")

In [ ]:
# 2 模型加载和推理
model = torch.load("model_best.pth")
model.eval()
print(f"modle: {model}")
dump_input = torch.tensor([[1,2]],dtype=torch.float32)
label = torch.matmul(dump_input, true_w) + true_b
print(f"true label:{label}")
pred = model(dump_input)
print(f"pred label:{pred}")


modle: LinearNet()
true label:tensor([-0.3000])
pred label:tensor([[-0.2994]], grad_fn=<AddBackward0>)


In [14]:
# 方法二：只保存模型权重
model_dict = model.state_dict()
torch.save(model_dict,"ckpt_best.pth")

In [15]:
# 模型加载和推理
model = LinearNet()
ckpt = torch.load("ckpt_best.pth")
model.load_state_dict(ckpt)
pred = model(dump_input)
print(f"pred: {pred}")

pred: tensor([[-0.2994]], grad_fn=<AddBackward0>)


| 特性                 | 方法一：保存整个模型                       | 方法二：只保存模型参数                    |
|----------------------|--------------------------------------------|-------------------------------------------|
| **实现难度**         | 简单，直接保存和加载                       | 需要手动定义模型架构                       |
| **文件大小**         | 大，包含架构和参数                         | 小，仅包含参数                              |
| **灵活性**           | 较低，架构和参数绑定                       | 高，架构和参数分离                         |
| **跨平台兼容性**     | 较差，受限于 PyTorch 和 Python 版本         | 较好，易与其他框架（如 ONNX）集成          |
| **适用场景**         | 快速验证、实验阶段                         | 生产部署、模型扩展                         |
| **重用性**           | 高，适合直接分发完整模型                   | 较低，需提供模型架构代码                   |
